# 01 — Supervised Learning: Data Loading, EDA, dan Evaluasi Klasifikasi

Dataset:
1. Bank Marketing — Binary Classification
2. Breast Cancer Wisconsin Diagnostic — Binary Classification
3. Reuters-21578 — Multilabel Classification

Notebook ini melakukan data loading, EDA, pembagian train-test, pelatihan baseline Logistic Regression, dan evaluasi pada data uji.


## 1. Instalasi dan import

In [ ]:
!pip -q install pandas numpy scikit-learn nltk


In [ ]:
import io, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, ConfusionMatrixDisplay, roc_auc_score,
    hamming_loss, jaccard_score
)
RANDOM_STATE = 42


## 2. Bank Marketing — Binary Classification

Sumber resmi: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing). Target `y` menunjukkan apakah nasabah berlangganan deposito (`yes`/`no`). Dataset `bank-full.csv` berisi 45.211 baris.


In [ ]:
bank_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"
with urllib.request.urlopen(bank_url) as response:
    bank_bytes = response.read()
with zipfile.ZipFile(io.BytesIO(bank_bytes)) as z:
    member = next(n for n in z.namelist() if n.endswith("bank-full.csv"))
    with z.open(member) as f:
        bank_df = pd.read_csv(f, sep=";")

print("Ukuran:", bank_df.shape)
display(bank_df.head())
display(bank_df.isna().sum().to_frame("missing"))
display(bank_df["y"].value_counts().to_frame("jumlah"))


### 2.1 Preprocessing, train-test split, dan model

Kolom `duration` dikeluarkan dari fitur karena durasi panggilan baru diketahui setelah panggilan berlangsung. Ini menghindari kebocoran informasi untuk skenario prediksi sebelum panggilan selesai.


In [ ]:
X_bank = bank_df.drop(columns=["y", "duration"])
y_bank = bank_df["y"].map({"no": 0, "yes": 1})

num_cols = X_bank.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_bank.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bank, y_bank, test_size=0.2, random_state=RANDOM_STATE, stratify=y_bank
)
bank_model = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1500, class_weight="balanced"))
])
bank_model.fit(X_train_b, y_train_b)
pred_b = bank_model.predict(X_test_b)
prob_b = bank_model.predict_proba(X_test_b)[:, 1]


### 2.2 Evaluasi klasifikasi Bank Marketing

In [ ]:
def evaluate_binary(name, y_true, y_pred, y_score):
    result = {
        "dataset": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score)
    }
    print(classification_report(y_true, y_pred, target_names=["no", "yes"], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["no", "yes"])
    plt.title("Confusion Matrix — " + name)
    plt.show()
    return result

bank_metrics = evaluate_binary("Bank Marketing", y_test_b, pred_b, prob_b)
display(pd.DataFrame([bank_metrics]))


## 3. Breast Cancer Wisconsin Diagnostic — Binary Classification

Sumber resmi: [UCI Breast Cancer Wisconsin Diagnostic](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic). Dataset ini memiliki 569 instance, 30 fitur numerik, dan target diagnosis `M` atau `B`.


In [ ]:
bc_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wdbc/wdbc.data"
bc_columns = ["id", "diagnosis"] + [f"feature_{i}" for i in range(1, 31)]
bc_df = pd.read_csv(bc_url, header=None, names=bc_columns)

print("Ukuran:", bc_df.shape)
display(bc_df.head())
display(bc_df["diagnosis"].value_counts().to_frame("jumlah"))
print("Missing values:", int(bc_df.isna().sum().sum()))


### 3.1 Pelatihan dan evaluasi

In [ ]:
X_bc = bc_df.drop(columns=["id", "diagnosis"])
y_bc = bc_df["diagnosis"].map({"B": 0, "M": 1})

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=RANDOM_STATE, stratify=y_bc
)
bc_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
bc_model.fit(X_train_c, y_train_c)
pred_c = bc_model.predict(X_test_c)
prob_c = bc_model.predict_proba(X_test_c)[:, 1]

bc_metrics = evaluate_binary("Breast Cancer Wisconsin", y_test_c, pred_c, prob_c)
display(pd.DataFrame([bc_metrics]))


## 4. Reuters-21578 — Multilabel Classification

Sumber: [UCI Reuters-21578](https://doi.org/10.24432/C52G6M) dan [dokumentasi Reuters Corpus NLTK](https://www.nltk.org/book/ch02.html).

**Perbedaan versi:** Arsip Reuters-21578 asli memiliki 21.578 dokumen. Corpus Reuters dalam NLTK merupakan subset yang umum digunakan (10.788 dokumen dan 90 topik). Kategori dapat tumpang tindih: satu berita bisa memiliki lebih dari satu label. Notebook ini menggunakan versi NLTK, sehingga laporkan jumlah aktual yang tercetak.


In [ ]:
import nltk
nltk.download("reuters", quiet=True)
from nltk.corpus import reuters

rows = []
for file_id in reuters.fileids():
    rows.append({
        "document_id": file_id,
        "text": reuters.raw(file_id),
        "labels": reuters.categories(file_id)
    })
reuters_df = pd.DataFrame(rows)
reuters_df["label_count"] = reuters_df["labels"].apply(len)

all_labels = sorted({label for labels in reuters_df["labels"] for label in labels})
print("Dokumen dalam corpus NLTK:", len(reuters_df))
print("Jumlah kategori unik:", len(all_labels))
print("Dokumen dengan >1 label:", int((reuters_df["label_count"] > 1).sum()))
display(reuters_df["label_count"].value_counts().sort_index().to_frame("jumlah_dokumen"))


### 4.1 Persiapan data multilabel

Corpus NLTK menyediakan pembagian `training/` dan `test/`. Label yang muncul kurang dari 5 kali pada data training dikeluarkan dari eksperimen model agar model memiliki contoh positif yang cukup. Statistik jumlah kategori di atas tetap dihitung dari seluruh corpus.


In [ ]:
train_docs = reuters_df[reuters_df["document_id"].str.startswith("training/")].copy()
test_docs = reuters_df[reuters_df["document_id"].str.startswith("test/")].copy()

label_counts = {}
for labels in train_docs["labels"]:
    for label in labels:
        label_counts[label] = label_counts.get(label, 0) + 1
model_labels = sorted([label for label, count in label_counts.items() if count >= 5])

def filter_labels(labels):
    return [label for label in labels if label in model_labels]

train_label_lists = train_docs["labels"].apply(filter_labels).tolist()
test_label_lists = test_docs["labels"].apply(filter_labels).tolist()

train_keep = [len(x) > 0 for x in train_label_lists]
test_keep = [len(x) > 0 for x in test_label_lists]
train_text = train_docs.loc[train_keep, "text"].tolist()
test_text = test_docs.loc[test_keep, "text"].tolist()
train_label_lists = [x for x in train_label_lists if x]
test_label_lists = [x for x in test_label_lists if x]

mlb = MultiLabelBinarizer(classes=model_labels)
Y_train = mlb.fit_transform(train_label_lists)
Y_test = mlb.transform(test_label_lists)

print("Label model:", len(model_labels))
print("Train:", len(train_text), "Test:", len(test_text))
print("Matriks label:", Y_train.shape, Y_test.shape)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

tfidf = TfidfVectorizer(
    stop_words="english", max_features=30000,
    min_df=2, max_df=0.95, sublinear_tf=True
)
X_train_r = tfidf.fit_transform(train_text)
X_test_r = tfidf.transform(test_text)

reuters_model = OneVsRestClassifier(
    LogisticRegression(solver="liblinear", max_iter=1000)
)
reuters_model.fit(X_train_r, Y_train)
pred_r = reuters_model.predict(X_test_r)
print("Jumlah fitur TF-IDF:", X_train_r.shape[1])


### 4.2 Evaluasi multilabel

- **Subset accuracy**: proporsi dokumen dengan seluruh himpunan label diprediksi tepat.
- **Micro-F1**: menghitung kontribusi semua keputusan label secara global.
- **Macro-F1**: menghitung F1 tiap label lalu merata-ratakannya.
- **Hamming loss**: proporsi keputusan label yang salah; semakin rendah semakin baik.
- **Jaccard samples**: kemiripan himpunan label aktual dan prediksi per dokumen.


In [ ]:
reuters_metrics = {
    "dataset": "Reuters-21578 (NLTK subset)",
    "subset_accuracy": accuracy_score(Y_test, pred_r),
    "micro_precision": precision_score(Y_test, pred_r, average="micro", zero_division=0),
    "micro_recall": recall_score(Y_test, pred_r, average="micro", zero_division=0),
    "micro_f1": f1_score(Y_test, pred_r, average="micro", zero_division=0),
    "macro_f1": f1_score(Y_test, pred_r, average="macro", zero_division=0),
    "hamming_loss": hamming_loss(Y_test, pred_r),
    "jaccard_samples": jaccard_score(Y_test, pred_r, average="samples", zero_division=0)
}
display(pd.DataFrame([reuters_metrics]))


## 5. Ringkasan hasil evaluasi

In [ ]:
binary_results = pd.DataFrame([bank_metrics, bc_metrics])
display(binary_results)
print("Evaluasi Reuters Multilabel:")
display(pd.DataFrame([reuters_metrics]))


## 6. Referensi

1. Moro, S., Rita, P., & Cortez, P. (2014). *Bank Marketing*. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306
2. Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). *Breast Cancer Wisconsin (Diagnostic)*. UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B
3. UCI Machine Learning Repository. *Reuters-21578 Text Categorization Collection*. https://doi.org/10.24432/C52G6M
4. NLTK Project. *Reuters Corpus*. https://www.nltk.org/book/ch02.html
5. scikit-learn developers. *Metrics and scoring: quantifying the quality of predictions*. https://scikit-learn.org/stable/modules/model_evaluation.html

Metrik dihasilkan oleh eksperimen pada notebook ini; nilainya dapat berubah jika data, preprocessing, pembagian data, atau parameter model diubah.
